# Introduction

The notebook contains testing of a new approach of fitting orbitrap spectra and related time estimations and optimization.

The new approach is basically a different way to perform segmentation of spectrum prior to fitting.
The spectrum is splitted by only-sero-containing slices.


# Modules


In [ ]:
from mascope.lib.file_func import load_file
from mascope.lib.peak import detect_peaks, segment_spec
import numpy as np
import json
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mascope.lib.file_func import get_instrument_type

# Instruments functions


In [ ]:
def resolution_function(mz):
    return 1.725e6 / np.sqrt(mz)


with open("C:/Users/alvai/Desktop/peak_shape.json", "r") as f:
    peak_shape = json.load(f)

# Existing fitting routine


In [ ]:
file_name = "KORBI2_20240328_1743_MION2_DBrMe_HCldil_HSI_100_Clean_420_40-500mz"
instrument_type = get_instrument_type(file_name)

In [ ]:
# Assign peak fitting threshold depending on the instrument type
# Correct intrument type unsured by get_instrument_type
if instrument_type == "orbi":
    threshold = 0.8
if instrument_type == "tof":
    threshold = 0.9

sample_file_data, all_peak_mzs = await detect_peaks(
    file_name,
    (peak_shape, resolution_function),
    threshold,
    # u_list = np.arange(50, 550, 1), # for testing
    max_n_peaks=5,
    if_exists="replace",
    dmz=0.5,
    return_peak_mzs=True,
    instrument_type="orbi",
)

# Separate window testing


In [ ]:
# Load spectrum
f = load_file(file_name)

In [ ]:
# mz value for testing
u = 63.98
# Window width
dmz = 0.1
# Get spectrum
mz = f.signal.mz
sum_spec = f.signal.interpolate_na(dim="mz", method="linear").sum(dim="time").compute()
mz = sum_spec.mz
# Cut window from spectrum
mz_win = mz.sel(mz=slice(u - dmz, u + dmz)).compute().values
spec_win = sum_spec.sel(mz=slice(u - dmz, u + dmz)).compute().values

plt.plot(mz_win, spec_win)
# plt.yscale("log")

# Test mz position precision for fitted peaks

Currently, the mz values of the fitted peaks are binded to the existing mz grid of the zarr file.
Here we compare how big is the difference between initial fitted peak positions and the binded to the mz grid.


In [ ]:
mz_init = all_peak_mzs
mz_rounded = f.peak_heights.dropna(dim="mz", how="all").mz.values
mz_diff = np.abs(mz_init - mz_rounded)
ppm_diff = 1e6 * mz_diff / mz_init


# peak_mz_mask = np.where(ppm_diff > 1)
# mz_rounded[peak_mz_mask] = mz_init[peak_mz_mask]
# mz_diff = np.abs(mz_init - mz_rounded)
# ppm_diff = 1e6 * mz_diff / mz_init


ppm_median = np.median(ppm_diff)
ppm_mean = ppm_diff.mean()

fig = plt.figure(figsize=(15, 3))

plt.scatter(mz_init, ppm_diff)
plt.axhline(
    ppm_median, color="red", ls="--", label=f"Median: {ppm_median:.3f}", linewidth=2
)
plt.axhline(
    ppm_mean, color="orange", ls="--", label=f"Mean: {ppm_mean:.3f}", linewidth=2
)

plt.legend()
plt.grid()
plt.title(
    "The difference caused by binding fitted peak positions to the existing mz grid"
)
plt.ylabel("Difference [ppm]")
plt.xlabel("m/z [Th]")

## Time estimations


"stupid" solution time estimation


In [ ]:
%%time
def slow_segmentation(spec):
    indices = []
    ind_chunk = []
    for i, val in enumerate(spec):
        if val == 0:
            if ind_chunk == []:
                continue
            else:
                indices.append(np.array([ind_chunk[0] - 1] + ind_chunk + [ind_chunk[-1] + 1], dtype=np.int64))
                ind_chunk = []
        else:
            ind_chunk.append(i)
    return indices

slow_segmentation(sum_spec);

"wise" solution time estimation


In [ ]:
%%time
segment_spec(sum_spec);

In [ ]:
slow_seg = slow_segmentation(spec_win)
fast_seg = segment_spec(spec_win)

fig, ax = plt.subplots(figsize=(15, 3), ncols=2)

ax[0].plot(mz_win, spec_win)
for i in slow_seg:
    ax[0].plot(mz_win[i], spec_win[i])
ax[0].set(yscale="log", title="Slow segmentation")


ax[1].plot(mz_win, spec_win)

for i in fast_seg:
    ax[1].plot(mz_win[i], spec_win[i])

ax[1].set(yscale="log", title="Fast segmentation")

fig.suptitle("Comparison of the outputs")

Check distribution of chunk lengths


In [ ]:
segment_ind = segment_spec(sum_spec)

chunk_lengths = [len(i) for i in segment_ind]
chunk_mzs = [np.mean(mz[i]) for i in segment_ind]
plt.scatter(chunk_mzs, chunk_lengths)
plt.ylabel("chunk width")
plt.xlabel("mz scale")

In [ ]:
peaks_per_segment = []

all_mzs = f.peak_heights.dropna(dim="mz", how="all").mz.values


for segment in segment_ind:

    mz_chunk = mz[segment].values
    num_of_peaks = len(
        f.peak_heights.mz[
            np.where((all_mzs > mz_chunk.min()) & (all_mzs < mz_chunk.max()))
        ]
    )

    peaks_per_segment.append(num_of_peaks)


plt.figure(figsize=(15, 3))


plt.scatter(chunk_mzs, peaks_per_segment)

plt.ylabel("Number of peaks per segment")

plt.xlabel("mz scale")

Check segments with n number of peaks


In [ ]:
set_num_peaks = 2

mz_num_segments = []
spec_num_segments = []
intensities = []

for segment in segment_ind:
    mz_chunk = mz[segment].values
    num_of_peaks = len(
        all_mzs[(all_mzs >= mz_chunk.min()) & (all_mzs <= mz_chunk.max())]
    )
    if num_of_peaks == set_num_peaks:
        # Filter with more than 2 zeros inside
        # if list(sum_spec[segment].values).count(0) == 3:
        mz_num_segments.append(mz_chunk)
        spec_num_segments.append(sum_spec[segment].values)
        intensities.append(sum_spec[segment].values.max())


fig = go.Figure()

for i, val in enumerate(mz_num_segments):
    fig.add_trace(go.Scatter(x=val, y=spec_num_segments[i], name=f"Segment {i}"))

fig.update_layout(title="Spectra Segments", xaxis_title="M/Z", yaxis_title="Intensity")

fig.show()

# Notes

Peak "wings" are not fitted optimally, likely because we can not describe the whole range of intensities with one median peak shape.
One of the options could be to perform separate peak shape estimations for different intensity ranges.

TOF spectra fitting routine must be improved.
Currently the only obvious thing is that baseline in spectrum should be taken into account.
